# 04 - Training Pipeline
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Dataset preparation (tokenization + feature extraction)
2. Pre-training the fusion model (neural network branches)
3. Training the XGBoost classifier on fused features
4. MLflow experiment tracking
5. Model checkpointing

In [ ]:
import sys
import os
import math
import re
import logging
from collections import Counter
from urllib.parse import urlparse

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import DistilBertModel, DistilBertTokenizer
import xgboost as xgb
import mlflow

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 4.1 Define All Model Components

Re-define model classes here so this notebook is self-contained and runnable.

In [ ]:
# ── Feature extraction ──
def _shannon_entropy(text: str) -> float:
    if not text:
        return 0.0
    freq = Counter(text)
    length = len(text)
    return -sum((c / length) * math.log2(c / length) for c in freq.values())

def _has_ip_address(hostname: str) -> int:
    return 1 if re.match(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$", hostname) else 0

def _max_consecutive_consonants(text: str) -> int:
    vowels = set("aeiouAEIOU")
    max_count = current = 0
    for char in text:
        if char.isalpha() and char not in vowels:
            current += 1
            max_count = max(max_count, current)
        else:
            current = 0
    return max_count

def _vowel_ratio(text: str) -> float:
    alpha_chars = [c for c in text if c.isalpha()]
    if not alpha_chars:
        return 0.0
    vowels = set("aeiouAEIOU")
    return sum(1 for c in alpha_chars if c in vowels) / len(alpha_chars)

def extract_url_features(url: str) -> dict:
    parsed = urlparse(url)
    hostname = parsed.hostname or ""
    path = parsed.path or ""
    return {
        "url_length": len(url), "hostname_length": len(hostname), "path_length": len(path),
        "num_dots": url.count("."), "num_hyphens": url.count("-"), "num_underscores": url.count("_"),
        "num_slashes": url.count("/"),
        "num_query_params": len(parsed.query.split("&")) if parsed.query else 0,
        "num_fragments": 1 if parsed.fragment else 0,
        "num_digits": sum(c.isdigit() for c in url),
        "num_special_chars": sum(not c.isalnum() and c not in ".-_/:" for c in url),
        "url_entropy": _shannon_entropy(url), "hostname_entropy": _shannon_entropy(hostname),
        "has_ip_address": _has_ip_address(hostname),
        "has_punycode": 1 if hostname.startswith("xn--") else 0,
        "has_port": 1 if parsed.port and parsed.port not in (80, 443) else 0,
        "has_https": 1 if parsed.scheme == "https" else 0,
        "has_at_symbol": 1 if "@" in url else 0,
        "has_double_slash_redirect": 1 if "//" in path else 0,
        "subdomain_count": len(hostname.split(".")) - 2 if len(hostname.split(".")) > 2 else 0,
        "tld_length": len(hostname.split(".")[-1]) if "." in hostname else 0,
        "consecutive_consonants_max": _max_consecutive_consonants(hostname),
        "vowel_ratio": _vowel_ratio(hostname),
    }

# ── Model classes ──
class AttentionLayer(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)
    def forward(self, lstm_output):
        weights = torch.softmax(self.attention(lstm_output), dim=1)
        return torch.sum(weights * lstm_output, dim=1)

class NLPBranch(nn.Module):
    def __init__(self, output_dim=128, lstm_hidden=256, lstm_layers=2, dropout=0.3, freeze_bert=True):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        if freeze_bert:
            for param in self.distilbert.parameters():
                param.requires_grad = False
        bert_hidden = self.distilbert.config.hidden_size
        self.bilstm = nn.LSTM(bert_hidden, lstm_hidden, lstm_layers, batch_first=True,
                              bidirectional=True, dropout=dropout if lstm_layers > 1 else 0)
        self.attention = AttentionLayer(lstm_hidden * 2)
        self.fc = nn.Linear(lstm_hidden * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, input_ids, attention_mask):
        bert_out = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        lstm_out, _ = self.bilstm(bert_out.last_hidden_state)
        attended = self.attention(lstm_out)
        return self.fc(self.dropout(attended))

class MLPBranch(nn.Module):
    def __init__(self, input_dim=24, hidden_dims=None, output_dim=64, dropout=0.3):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128, 64]
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)])
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

class PhishScamSenseFusionModel(nn.Module):
    def __init__(self, num_features=24, nlp_output_dim=128, numerical_output_dim=64, freeze_bert=True):
        super().__init__()
        self.nlp_branch = NLPBranch(output_dim=nlp_output_dim, freeze_bert=freeze_bert)
        self.numerical_branch = MLPBranch(input_dim=num_features, output_dim=numerical_output_dim)
        self.fusion_dim = nlp_output_dim + numerical_output_dim
    def forward(self, input_ids, attention_mask, numerical_features):
        nlp_out = self.nlp_branch(input_ids, attention_mask)
        num_out = self.numerical_branch(numerical_features)
        return torch.cat([nlp_out, num_out], dim=1)

class URLTokenizer:
    def __init__(self, max_length=128):
        self.tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
        self.max_length = max_length
    def tokenize(self, urls):
        return self.tokenizer(urls, padding=True, truncation=True,
                              max_length=self.max_length, return_tensors="pt")

print("All model components defined.")

## 4.2 Prepare Dataset

Tokenize URLs for the NLP branch and extract numerical features for the MLP branch.

In [ ]:
# Sample URLs (replace with your actual dataset)
urls = [
    # Benign
    "https://www.google.com/search?q=python",
    "https://github.com/anthropics/claude-code",
    "https://stackoverflow.com/questions/tagged/python",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    "https://docs.python.org/3/library/urllib.html",
    "https://www.amazon.com/dp/B08N5WRWNW",
    "https://www.reddit.com/r/MachineLearning",
    "https://mail.google.com/mail/u/0/#inbox",
    "https://www.linkedin.com/in/johndoe",
    "https://www.microsoft.com/en-us/windows",
    "https://www.apple.com/macbook-pro",
    "https://www.netflix.com/browse",
    "https://twitter.com/home",
    "https://www.bbc.com/news/world",
    # Phishing
    "http://192.168.1.1/login/google-verify.html",
    "http://xn--ggle-1noa.com/accounts/login",
    "http://googl3-security.com/verify?user=admin&token=abc123",
    "http://paypa1-secure.com/signin/update-billing",
    "http://amaz0n-support.xyz/account/verify",
    "http://microsoft-365-login.tk/auth/signin",
    "http://netflix-billing-update.ml/payment",
    "http://faceb00k-security.ga/hacked/recovery",
    "http://apple-id-verify.cf/icloud/login.php",
    "http://bank0famerica-secure.ru/online/login",
    "http://dhl-tracking-update.info/parcel?id=83927492",
    "http://instagram-verify-account.net/auth",
    "http://linkedln-security.com/checkpoint/verify",
    "http://dropbox-shared-doc.tk/dl/invoice.pdf.exe",
    "http://wellsfarg0-alert.com/security/update",
]
labels = [0]*15 + [1]*15

def prepare_dataset(urls, labels):
    """Prepare dataset with both NLP tokens and numerical features."""
    tokenizer = URLTokenizer()
    tokens = tokenizer.tokenize(urls)

    numerical_features = []
    for url in urls:
        features = extract_url_features(url)
        numerical_features.append(list(features.values()))

    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
        "numerical_features": torch.tensor(numerical_features, dtype=torch.float32),
        "labels": np.array(labels),
    }

data = prepare_dataset(urls, labels)
print(f"input_ids shape:          {data['input_ids'].shape}")
print(f"attention_mask shape:     {data['attention_mask'].shape}")
print(f"numerical_features shape: {data['numerical_features'].shape}")
print(f"labels shape:             {data['labels'].shape}")

## 4.3 Train Fusion Model (Neural Network Branches)

Pre-train the neural network with BCEWithLogitsLoss to learn good feature representations before extracting features for XGBoost.

In [ ]:
def train_fusion_model(fusion_model, train_data, epochs=10, batch_size=8, learning_rate=1e-4, device="cpu"):
    """Pre-train the neural network branches with supervised loss."""
    fusion_model = fusion_model.to(device)
    fusion_model.train()

    classifier_head = nn.Linear(fusion_model.fusion_dim, 1).to(device)
    optimizer = torch.optim.Adam(
        list(fusion_model.parameters()) + list(classifier_head.parameters()),
        lr=learning_rate,
    )
    criterion = nn.BCEWithLogitsLoss()

    dataset = TensorDataset(
        train_data["input_ids"],
        train_data["attention_mask"],
        train_data["numerical_features"],
        torch.tensor(train_data["labels"], dtype=torch.float32),
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    history = {"loss": []}

    for epoch in range(epochs):
        total_loss = 0
        for input_ids, attention_mask, num_feat, batch_labels in dataloader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            num_feat = num_feat.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            fused = fusion_model(input_ids, attention_mask, num_feat)
            logits = classifier_head(fused).squeeze(-1)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        history["loss"].append(avg_loss)
        logger.info(f"Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}")

    return fusion_model, history

print("Training function defined.")

## 4.4 Run Training with MLflow Tracking

In [ ]:
import matplotlib.pyplot as plt

# MLflow tracking URI (local filesystem if server not running)
MLFLOW_TRACKING_URI = os.path.join(PROJECT_ROOT, "ml", "mlruns")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("phishscamsense")

EPOCHS = 5   # increase for real training
BATCH_SIZE = 8
LR = 1e-4

with mlflow.start_run(run_name="fusion_model_v1"):
    mlflow.log_params({
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "model_type": "DistilBERT+BiLSTM+Attention+MLP+XGBoost",
        "nlp_output_dim": 128,
        "numerical_output_dim": 64,
        "freeze_bert": True,
        "num_samples": len(urls),
    })

    # Step 1: Train fusion model
    print("Step 1: Training fusion model (neural network branches)...")
    fusion_model = PhishScamSenseFusionModel(num_features=24, freeze_bert=True)
    fusion_model, history = train_fusion_model(
        fusion_model, data, epochs=EPOCHS, batch_size=BATCH_SIZE,
        learning_rate=LR, device=device,
    )
    for step, loss_val in enumerate(history["loss"]):
        mlflow.log_metric("train_loss", loss_val, step=step)

    # Step 2: Extract fused features for XGBoost
    print("\nStep 2: Extracting fused features for XGBoost...")
    fusion_model.eval()
    with torch.no_grad():
        fused_features = fusion_model(
            data["input_ids"].to(device),
            data["attention_mask"].to(device),
            data["numerical_features"].to(device),
        ).cpu().numpy()
    print(f"Fused feature matrix shape: {fused_features.shape}")

    # Step 3: Train XGBoost
    print("\nStep 3: Training XGBoost classifier...")
    xgb_clf = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        objective="binary:logistic", eval_metric="logloss",
        use_label_encoder=False,
    )
    xgb_clf.fit(fused_features, data["labels"])

    # Step 4: Training accuracy
    predictions_proba = xgb_clf.predict_proba(fused_features)[:, 1]
    predictions = (predictions_proba > 0.5).astype(int)
    accuracy = np.mean(predictions == data["labels"])
    mlflow.log_metric("train_accuracy", accuracy)

    print(f"\nTraining accuracy: {accuracy:.4f}")
    print(f"MLflow run logged to: {MLFLOW_TRACKING_URI}")

print("\nTraining complete!")

## 4.5 Plot Training Loss & Save Checkpoints

In [ ]:
# Plot training loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(history["loss"]) + 1), history["loss"], marker="o", color="#3498db")
ax.set_title("Fusion Model Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCEWithLogitsLoss")
ax.grid(True)
plt.tight_layout()
plt.show()

# Save model checkpoints
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "ml", "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Save fusion model (PyTorch)
torch.save(fusion_model.state_dict(), os.path.join(CHECKPOINT_DIR, "fusion_model.pt"))

# Save XGBoost model
import pickle
with open(os.path.join(CHECKPOINT_DIR, "xgb_classifier.pkl"), "wb") as f:
    pickle.dump(xgb_clf, f)

print(f"Checkpoints saved to: {CHECKPOINT_DIR}")
print(f"  - fusion_model.pt")
print(f"  - xgb_classifier.pkl")